In [1]:
from openpyxl import load_workbook
import pandas as pd
import unicodedata
import re

In [2]:
df = pd.read_excel('Libro1.xlsx')
df.columns = df.columns.str.strip()

wb = load_workbook('Libro1.xlsx')
ws = wb['BASE DE DATOS']

fecha_inicio = 'Fecha de Inicio del Contrato de Interventoría'
fecha_fin = 'Fecha de Finalización del Contrato de Interventoría'

headers = [
    cell.value.strip() if isinstance(cell.value, str) else cell.value
    for cell in ws[1]
]

In [3]:
col_inicio = headers.index(fecha_inicio) + 1
col_fin = headers.index(fecha_fin) + 1

def tiene_color(celda):
    fill = celda.fill

    if fill is None:
        return False

    if fill.fill_type is None:
        return False

    color = fill.fgColor.rgb

    if color in [None, '00000000', 'FFFFFFFF']:
        return False

    return True

estados = []

for row in range(2, ws.max_row + 1):
    celda_inicio = ws.cell(row=row, column=col_inicio)
    celda_fin = ws.cell(row=row, column=col_fin)

    if tiene_color(celda_inicio) or tiene_color(celda_fin):
        estados.append('FINALIZO')
    else:
        estados.append('ACTIVO')

df['Contrato Estado'] = estados

In [4]:

df = df.dropna(
    subset=[
        'Nombres y Apellidos',
        'No. Cedula de ciudadanía',
        'No. Contrato de Interventoría'
    ],
    how='all'
)

In [5]:
df['Nombres y Apellidos'] = (
    df['Nombres y Apellidos']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)
df['Nombres y Apellidos'] = df['Nombres y Apellidos'].apply(
    lambda x: ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    ) if pd.notna(x) else x
)
df['Nombres y Apellidos'] = df['Nombres y Apellidos'].str.replace(r'[^A-ZÑ ]', '', regex=True)

In [6]:
df['No. Cedula de ciudadanía'] = (
    df['No. Cedula de ciudadanía']
    .astype('string')
    .str.replace(r'\D', '', regex=True)
)
df['No. Cedula de ciudadanía'] = df['No. Cedula de ciudadanía'].replace('', pd.NA)
df['No. Cedula de ciudadanía'] = pd.to_numeric(
    df['No. Cedula de ciudadanía'],
    errors='coerce'
)

In [7]:
col = 'No. Contrato de Interventoría'

df[col] = (
    df[col]
    .astype('string')
    .str.upper()
    .str.replace(r'-20\d{2}$', '', regex=True)
    .str.replace(r'\bNO\.\s*', '', regex=True)
    .str.replace('-', '', regex=False)
    .str.replace(r'\D', '', regex=True)
)

df[col] = df[col].fillna('').replace('', '0')
df.loc[df[col].str.len() < 3, col] = '0'
df.loc[df[col].str.fullmatch(r'0+', na=False), col] = '0'

In [8]:
df['Año del Contrato de Interventoría'] = (
    df['Año del Contrato de Interventoría']
    .astype('string')
    .str.extract(r'(19\d{2}|20\d{2})')[0]
    .fillna('0')
)

In [9]:
FECHA_VACIA = '0000-00-00'

MESES = {
    'ENERO': '01', 'FEBRERO': '02', 'MARZO': '03', 'ABRIL': '04',
    'MAYO': '05', 'JUNIO': '06', 'JULIO': '07', 'AGOSTO': '08',
    'SEPTIEMBRE': '09', 'SETIEMBRE': '09', 'OCTUBRE': '10',
    'NOVIEMBRE': '11', 'DICIEMBRE': '12',
}

MESES_CORTO = {
    'ENE': '01', 'FEB': '02', 'MAR': '03', 'ABR': '04',
    'MAY': '05', 'JUN': '06', 'JUL': '07', 'AGO': '08',
    'SEP': '09', 'OCT': '10', 'NOV': '11', 'DIC': '12',
}

RE_SOLO_TEXTO = re.compile(r'[A-ZÁÉÍÓÚÑ\s,\.]+')
RE_ANIO_MALO = re.compile(r'^0(\d{3})-(\d{2})-(\d{2})$')
RE_FECHA_ISO = re.compile(r'(\d{4}-\d{2}-\d{2})')
RE_FECHA_SLASH = re.compile(r'(\d{1,2})/(\d{1,2})/\(?(\d{4})')
RE_FECHA_DOBLE_SLASH = re.compile(r'(\d{1,2})/(\d{1,2})//(\d{2})')
RE_TEXTO_CON_FECHA = re.compile(r'(\d{1,2})\s+(?:DE\s+)?([A-ZÁÉÍÓÚÑ]+)\s+(?:DE|DEL)?\s+(\d{4})')
RE_FECHA_CORTA = re.compile(r'(\d{1,2})-([A-ZÁÉÍÓÚÑ]{3})-(\d{4})')
RE_MES_ANIO = re.compile(r'^([A-ZÁÉÍÓÚÑ]+)\s+DE\s+(\d{4})$')


def convertir_fecha_texto(valor):
    if pd.isna(valor):
        return FECHA_VACIA

    valor = re.sub(r'\s+', ' ', str(valor).upper().strip())

    if valor in ('00/00/0000', '-'):
        return FECHA_VACIA

    # Si solo tiene letras, espacios, comas o puntos, no es una fecha
    if RE_SOLO_TEXTO.fullmatch(valor):
        return FECHA_VACIA

    # 0201-01-28 -> 2001-01-28
    m = RE_ANIO_MALO.search(valor)
    if m:
        anio, mes, dia = m.groups()
        return f'2{anio}-{mes}-{dia}'

    # 2016-07-28 00:00:00 -> 2016-07-28
    m = RE_FECHA_ISO.search(valor)
    if m:
        return m.group(1)

    # FECHA PROBABLE 08/12/2021 / REINICIO 08/08/2022 / 09/12/(2019
    m = RE_FECHA_SLASH.search(valor)
    if m:
        dia, mes, anio = m.groups()
        return f'{anio}-{mes.zfill(2)}-{dia.zfill(2)}'

    # 17/03//21 -> 2021-03-17
    m = RE_FECHA_DOBLE_SLASH.search(valor)
    if m:
        dia, mes, anio = m.groups()
        return f'20{anio}-{mes.zfill(2)}-{dia.zfill(2)}'

    # 07 ENERO DE 2020 / 14 OCTUBRE DEL 2022 / 1 DE DICIEMBRE 2022
    m = RE_TEXTO_CON_FECHA.search(valor)
    if m:
        dia, mes_txt, anio = m.groups()
        mes = MESES.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-{dia.zfill(2)}'

    # ESTIMADO 6-FEB-2023
    m = RE_FECHA_CORTA.search(valor)
    if m:
        dia, mes_txt, anio = m.groups()
        mes = MESES_CORTO.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-{dia.zfill(2)}'

    # FEBRERO DE 2019 -> 2019-02-00
    m = RE_MES_ANIO.search(valor)
    if m:
        mes_txt, anio = m.groups()
        mes = MESES.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-00'

    # Todo lo que no se pudo convertir queda como fecha por defecto
    return FECHA_VACIA


df['Fecha de Inicio del Contrato de Interventoría'] = (
    df['Fecha de Inicio del Contrato de Interventoría']
    .apply(convertir_fecha_texto)
)

df['Fecha de Finalización del Contrato de Interventoría'] = (
    df['Fecha de Finalización del Contrato de Interventoría']
    .apply(convertir_fecha_texto)
)

In [10]:
df.to_excel('interventoria_limpio.xlsx', index=False)